# MFDB Burst Selection Round-Trip Example

This notebook demonstrates how to interact with the **Multiparameter Fluorescence Database (MFDB)** and the **Burst Selection** plugin in-process using the high-level `BurstPipeline` wrapper.

### Database Connection & Storage Details:
- **Storage**: SQLite database file (defaults to `chisurf/core/fio/mmcif/db/sample_management.db`).
- **Explicit Connection**: We instantiate the database connection directly in Python, giving the user control over the target SQLite file path.
- **In-process Access**: Direct Python database connection (no server, client, or ZMQ network sockets needed).

### Workflow Steps:
1. Connect to the database explicitly.
2. Initialize the pipeline using the database connection.
3. Run burst selection analysis with custom parameters (registers input file, runs detection, and registers output processed burst data).
4. Query the database to retrieve the full provenance graph lineage.
5. Generate diagnostic plots (photon count rate, burst sizes/widths, and FRET histograms).

## 1. Connect to Database and initialize the Pipeline

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))  # Add package root to path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import chisurf
from chisurf.core.mfdb import FluorescenceDatabase, BurstPipeline

# Locate the example dataset
chisurf_dir = Path(chisurf.__file__).resolve().parent
spc_file = chisurf_dir / "plugins" / "burst" / "burst_selection" / "tests" / "data" / "bh_spc132_sm_dna" / "m000.spc"
print(f"Using SPC file: {spc_file}")

# Connect to the database explicitly
# (Pass a custom file path to connect to another DB, e.g. FluorescenceDatabase('my_data.db'))
db = FluorescenceDatabase()
print(f"Connected to database at: {db.db_path}")

# Initialize the pipeline with the database connection
pipeline = BurstPipeline(db)
print("Database pipeline initialized.")

## 2. Run Burst Selection analysis

In [ ]:
print("Running burst detection...")
bur_file_path = pipeline.run(spc_file, min_photons=20, time_window=1e-3)
print(f"Burst selection complete. Output .bur path: {bur_file_path}")

## 3. Query MFDB graph lineage

In [ ]:
edges = pipeline.get_lineage()
print("Upstream graph lineage:")
for edge in edges:
    print(f"  {edge['source_node_type']}:{edge['source_node_id']} --[{edge['relationship_type']}]--> {edge['target_node_type']}:{edge['target_node_id']}")

## 4. Generate diagnostic plots from loaded data

In [ ]:
# Load TTTR file and calculate photon count rate over time
from chisurf.plugins.burst.burst_selection.api.io import load_tttr
tttr = load_tttr(spc_file)
macro_times = tttr.macro_times
resolution = tttr.header.macro_time_resolution
time_seconds = macro_times * resolution

# Bin time into 100ms intervals to get intensity trace
bin_width = 0.1
bins = np.arange(time_seconds[0], time_seconds[-1] + bin_width, bin_width)
counts, edges = np.histogram(time_seconds, bins=bins)
bin_centers = (edges[:-1] + edges[1:]) / 2.0
count_rate = counts / bin_width

# Plot intensity trace
plt.figure(figsize=(12, 4))
plt.plot(bin_centers, count_rate / 1000.0, color='darkblue', alpha=0.7)
plt.xlabel("Time (s)")
plt.ylabel("Photon Count Rate (kHz)")
plt.title("Example TTTR Photon Count Rate (Intensity Trace)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Read output burst table and extract features
from chisurf.plugins.burst.burst_selection.api.features import extract_features
df_raw = pd.read_csv(bur_file_path, sep='\t')
df_bur = extract_features([df_raw])
df_bur = df_bur[df_bur['nphotons'] > 0]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. Burst size (nphotons) histogram
axes[0].hist(df_bur['nphotons'], bins=50, color='forestgreen', edgecolor='black', alpha=0.7, log=True)
axes[0].set_xlabel("Burst Size (n_photons)")
axes[0].set_ylabel("Frequency (log scale)")
axes[0].set_title("Burst Size Distribution")
axes[0].grid(True, linestyle=':', alpha=0.6)

# 2. Burst duration histogram
axes[1].hist(df_bur['duration'], bins=50, color='crimson', edgecolor='black', alpha=0.7)
axes[1].set_xlabel("Burst Duration (ms)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Burst Duration Distribution")
axes[1].grid(True, linestyle=':', alpha=0.6)

# 3. FRET (Proximity Ratio) histogram
fret_col = 'fret' if 'fret' in df_bur.columns else 'Proximity Ratio'
n, bins_fret, patches = axes[2].hist(df_bur[fret_col], bins=50, range=(0.0, 1.0), density=True,
                                     color='purple', edgecolor='black', alpha=0.5, label='Data')

axes[2].set_xlabel("FRET Efficiency / Proximity Ratio")
axes[2].set_ylabel("Density")
axes[2].set_title("FRET Distribution")
axes[2].legend()
axes[2].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()